# Lecture 05: HTTP and Interfaces

This notebook sends real requests to a backend and prints what travels. Each section belongs to one slide of the lecture. Run the cells from the top.

The notebook starts the small backend from the slides in the background. An application has three fields, `age`, `monthly_income` and `utilisation`. The backend has no model. It approves every application, and its answer repeats the application as the backend received it. You can read it in `small_backend.py`.

In [ ]:
# Installs the packages of this notebook into the kernel you picked. It takes a moment the first time.
import sys

!uv pip install --quiet --python "{sys.executable}" "fastapi[standard]" httpx

## 1. Start the small backend

In [ ]:
import json
import socket
import time
from urllib.parse import urlsplit

import httpx

import small_backend
from small_backend import SAMPLE_APPLICATION

URL = small_backend.start_in_background()
print("The small backend answers at", URL)

## 2. A request is plain text

The next cell writes a request by hand and sends it through a plain network connection. No HTTP library is involved. What comes back is text as well.

In [ ]:
address = urlsplit(URL)
body = json.dumps(SAMPLE_APPLICATION)
request_text = (
    "POST /decisions HTTP/1.1\r\n"
    f"Host: {address.netloc}\r\n"
    "Content-Type: application/json\r\n"
    f"Content-Length: {len(body.encode())}\r\n"
    "Connection: close\r\n"
    "\r\n"
    f"{body}"
)

with socket.create_connection((address.hostname, address.port)) as connection:
    connection.sendall(request_text.encode())
    response_text = b""
    while chunk := connection.recv(4096):
        response_text += chunk

print("What we sent\n", request_text, sep="\n")
print("\n\nWhat came back\n", response_text.decode(), sep="\n")

The first line of the request names the method and the path. The headers follow, one per line. An empty line separates them from the body. The response has the same three parts, with the status in its first line.

Everything the backend knows about the request is in that text.

## 3. The same request with a library

Nobody writes requests by hand in practice. `httpx` builds the same text from a Python dictionary and turns the answer back into one. The rest of the notebook uses it.

In [ ]:
client = httpx.Client(base_url=URL, timeout=10)

answer = client.post("/decisions", json=SAMPLE_APPLICATION)
print(answer.status_code, answer.reason_phrase)
answer.json()

## 4. Predict the status code

The next cell sends six requests to the small backend. Write down the status code you expect for each one before you run the cell. The slide "Status Codes Come in Families" names every code you need.

In [ ]:
six_requests = [
    ("GET", "/health", None),
    ("POST", "/decisions", SAMPLE_APPLICATION),
    ("GET", "/loans", None),
    ("POST", "/decisions", SAMPLE_APPLICATION | {"age": "thirty"}),
    ("POST", "/decisions", SAMPLE_APPLICATION | {"age": 15}),
    ("DELETE", "/decisions", None),
]
for number, (method, path, body) in enumerate(six_requests, 1):
    answer = client.request(method, path, json=body)
    sent = "" if body is None else json.dumps(body)
    print(f"{number}. {method:<6} {path:<10} {sent:<66} {answer.status_code} {answer.reason_phrase}")

Request 4 sent an age of "thirty". The answer names the field and the reason. The backend refused the request before any of its own code ran.

In [ ]:
client.post("/decisions", json=SAMPLE_APPLICATION | {"age": "thirty"}).json()

## 5. Left out, null and 0

The next three requests differ only in the income. In the first it is left out, in the second it is `null`, and in the third it is zero. Look at what the backend received.

In [ ]:
without_income = {"age": 34, "utilisation": 0.25}
incomes = {
    "left out": without_income,
    "null": without_income | {"monthly_income": None},
    "0": without_income | {"monthly_income": 0},
}
for label, application in incomes.items():
    answer = client.post("/decisions", json=application)
    print(f"income {label:<9} {answer.status_code}   received {answer.json()['received']}")

A left-out income and a `null` arrive the same, because the contract gives `monthly_income` a default of `None`. Zero arrives as an income of zero. Nobody downstream can tell it apart from an applicant who earns nothing.

A field without a default behaves differently. Leave out the age, and the request does not get through at all.

In [ ]:
answer = client.post("/decisions", json={"monthly_income": 6000, "utilisation": 0.25})
print(answer.status_code, answer.reason_phrase)
answer.json()["detail"]

## 6. 30 or 0.30

Utilisation is a share, so 30 percent is 0.30. A person who types 30 means the same thing. Send both.

In [ ]:
for typed in (0.30, 30):
    answer = client.post("/decisions", json=SAMPLE_APPLICATION | {"utilisation": typed})
    utilisation = answer.json()["received"]["utilisation"]
    print(f"utilisation {typed:<5} {answer.status_code}   received {utilisation}, that is {utilisation:.0%}")

Both requests answer 200. The contract only asks for a value of at least zero, and 30 is one. The backend now works with a utilisation of 3000 percent, and nothing reports an error.

## 7. A network call can fail

A function call always arrives. A network call can fail in two ways that a function call cannot. Nobody may be listening, or the other side may be too slow.

The next cell starts a second small server that takes five seconds to answer, much like a language model writing a long text. It is a few lines of FastAPI.

In [ ]:
from fastapi import FastAPI

slow = FastAPI()


@slow.post("/wording")
def wording() -> dict[str, str]:
    time.sleep(5)
    return {"text": "We cannot grant the credit at present."}


SLOW_URL = small_backend.start_in_background(slow)
NOBODY_URL = f"http://127.0.0.1:{small_backend.free_port()}"

In [ ]:
calls = [
    ("nobody listens", NOBODY_URL, 2),
    ("too slow for the timeout", SLOW_URL, 2),
    ("slow, but within the timeout", SLOW_URL, 10),
]
for label, url, timeout in calls:
    started = time.perf_counter()
    try:
        answer = httpx.post(f"{url}/wording", timeout=timeout)
        outcome = f"{answer.status_code} {answer.json()}"
    except httpx.HTTPError as error:
        outcome = type(error).__name__
    print(f"{label:<30} timeout {timeout:>2} s   after {time.perf_counter() - started:4.1f} s   {outcome}")

The first failure comes at once. The second comes only when the timeout runs out.

`httpx` waits five seconds when you set no timeout. The `requests` library waits forever unless you set one. A page that calls a backend therefore says how long it waits, and what it shows when that time runs out.

## 8. Two small pages

The last two demos are pages, not cells. Run them in a terminal from this folder.

`rerun_demo.py` shows that Streamlit runs the whole script on every click.

```bash
uv run streamlit run rerun_demo.py
```

`call_demo.py` is the page from the slide "The Page Calls the Backend". Start the small backend first, then the page.

```bash
uv run uvicorn small_backend:app --port 8000
BACKEND_URL=http://localhost:8000 uv run streamlit run call_demo.py
```

Stop the backend while the page is open, and press Decide again.